In [1]:
import os
# CUDA_VISIBLE_DEVICES = "0,1,2,3,4,5,6,7"
# CUDA_VISIBLE_DEVICES = "0,1,2,3"
# CUDA_VISIBLE_DEVICES = "0,1"
CUDA_VISIBLE_DEVICES = "0, 1"
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['JAX_PLATFORM_NAME'] = 'gpu'

NPROC = len(CUDA_VISIBLE_DEVICES.split(","))

In [2]:
# import shutup
# shutup.please()

import warnings
warnings.filterwarnings('ignore')

# import rootutils
# ROOT = rootutils.setup_root(indicator='README.md', search_from=os.path.abspath(''), pythonpath=True, cwd=True)

import jax
import jax.numpy as jnp
import optax
from jaxtyping import ArrayLike, Float
import numpy as np
GLOBAL_KEY = jax.random.key(42)

import seaborn as sns
import matplotlib.pyplot as plt
# plt.style.use(['science', 'notebook'])

# Lagrangian Potentials

from ott2.neural.methods.lagrangian.lagrangian_potentials import *

%load_ext autoreload
%autoreload 2


In [3]:
## Utils

def draw_trajs(trajs: Float[ArrayLike, 'timestep point dim=2'], ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    colors = sns.color_palette("pastel", trajs.shape[1])
    
    for point in range(trajs.shape[1]):
        for t in range(1, trajs.shape[0]):
            ax.plot([trajs[t-1, point, 0], trajs[t, point, 0]],
                    [trajs[t-1, point, 1], trajs[t, point, 1]],
                    color=colors[point], linestyle="-", linewidth=1, marker='o', alpha=0.6, markersize=1)
    return ax

In [4]:
# Multigpu utils
from jax.sharding import Mesh
from jax.sharding import PartitionSpec
from jax.sharding import NamedSharding
from jax.experimental import mesh_utils

P = PartitionSpec
mesh = Mesh(mesh_utils.create_device_mesh((NPROC,)), axis_names=('data',))

def with_mesh(f):
    def wrapper(*args, **kwargs):
        with mesh:
            return f(*args, **kwargs)
    return wrapper

In [5]:
from torch.utils.data import Dataset, DataLoader

class InfiniteLoaderWrapper:
    def __init__(self, loader: DataLoader):
        self.loader = loader
        self.loader_it = iter(loader)
    
    def __iter__(self):
        self.loader_it = iter(self.loader)
        return self

    def __next__(self):
        try:
            batch = next(self.loader_it)
        except StopIteration:
            self.loader_it = iter(self.loader)
            batch = next(self.loader_it)
        return batch

class OTLoader:
    def __init__(
        self,
        src_ds: Dataset,
        trg_ds: Dataset,
        flatten_flag: bool = False,
        **torch_dataloader_kwargs,
    ):
        def collate_fn(batch: tuple[np.ndarray]):
            return np.stack(batch)

        self.src_loader = InfiniteLoaderWrapper(DataLoader(src_ds, collate_fn=collate_fn, **torch_dataloader_kwargs))
        self.trg_loader = InfiniteLoaderWrapper(DataLoader(trg_ds, collate_fn=collate_fn, **torch_dataloader_kwargs))
        self.flatten_flag = flatten_flag

    def __iter__(self):
        self.src_loader = iter(self.src_loader)
        self.trg_loader = iter(self.trg_loader)
        return self

    def __next__(self):
        src_batch = jnp.asarray(next(self.src_loader))
        tgt_batch = jnp.asarray(next(self.trg_loader))
        if self.flatten_flag:
            b_size = src_batch.shape[0]
            src_batch = src_batch.reshape(b_size, -1)
            tgt_batch = tgt_batch.reshape(b_size, -1)
        return {
            "src_lin": src_batch,
            "tgt_lin": tgt_batch,
        }

# Neural

In [6]:
import flax.linen as nn

In [27]:
class SinusoidalTimeEmbedding(nn.Module):
    embedding_dim: int = 128
    max_freq: float = 32.

    @nn.compact
    def __call__(self, timesteps):
        """
        timesteps: jnp.ndarray of shape (batch_size,) with int or float timesteps
        max_period: controls the minimum frequency of the embeddings
        Returns:
            jnp.ndarray of shape (batch_size, embedding_dim)
        """
        half_dim = self.embedding_dim // 2
        freq_factors = np.linspace(1., self.max_freq, num=half_dim)
        # Expand to shape (batch_size, half_dim)
        angles = 2 * jnp.pi * timesteps * freq_factors
        emb = jnp.concatenate([jnp.sin(angles), jnp.cos(angles)], axis=-1)
        return emb  # shape: (batch_size, embedding_dim)
        
class TimeEmbeddingMLP(nn.Module):
    embedding_dim: int
    out_dim: int

    @nn.compact
    def __call__(self, timesteps):
        x = SinusoidalTimeEmbedding(self.embedding_dim)(timesteps)
        x = nn.Dense(self.out_dim)(x)
        x = nn.silu(x)
        x = nn.Dense(self.out_dim)(x)
        return x

In [35]:
class MLP(nn.Module):
    hidden_layers: list
    t_embedding_dim: int = 32
    
    @nn.compact
    def __call__(self, t, x, _):
        t_emb = TimeEmbeddingMLP(
            embedding_dim=self.t_embedding_dim,
            out_dim=x.shape[-1],
        )(t)
        x = jnp.concatenate([x, t, t_emb], -1)
        for i, dim in enumerate(self.hidden_layers):
            x = nn.Dense(dim)(x)
            if i != len(self.hidden_layers) - 1:
                x = nn.leaky_relu(x)
        return x

In [36]:
from ott2.neural.networks.resnet_d import ResNet_D


class ResNetDwTime(nn.Module):
    size: int = 64 
    nlayers: int = 4
    nc: int = 3
    nfilter: int = 64
    nfilter_max: int = 512
    t_embedding_dim: int = 128

    @nn.compact
    def __call__(self, t, x, train=True):
        b_size = t.shape[0]

        x = x.reshape(-1, self.nc, self.size, self.size)

        # t_emb = nn.Dense(self.size**2)(t[:, None]) # [b, size**2]
        t_emb = TimeEmbeddingMLP(
            embedding_dim=self.t_embedding_dim,
            out_dim=self.size**2
        )(t[:, None]) # [b, size**2]
        t_emb = t_emb.reshape(b_size, 1, self.size, self.size)

        x_with_t = jnp.concatenate([x, t_emb], axis=1)
        return ResNet_D(
            size=self.size,
            nlayers=self.nlayers,
            nc=self.nc+1,
            nfilter=self.nfilter,
            nfilter_max=self.nfilter_max,
        )(x_with_t)


In [ ]:
from functools import partial
from flax.training.train_state import TrainState

@with_mesh
@partial(jax.jit, static_argnums=(0, 1))
def create_state(module: nn.Module, dim: int):
    variables = module.init(
        jax.random.PRNGKey(0),
        np.random.randn(1, 1),
        np.random.randn(1, dim),
        np.random.randn(1, dim),
    )
    state = TrainState.create(
        apply_fn=module.apply,
        params=variables["params"],
        tx=optax.adamw(3e-4),
    )
    state = jax.tree_map(jnp.asarray, state)
    state_spec = nn.get_partition_spec(state)
    state = jax.lax.with_sharding_constraint(state, state_spec)
    return state

module = MLP(hidden_layers=(128, 1))
state = create_state(module, dim=2)
jax.tree_map(jnp.shape, state)
jax.debug.visualize_array_sharding(state.params["Dense_0"]["kernel"])

# Neural Optimal Control

In [38]:
import numpy as np

from ott2.neural.methods.nocc import NeuralOC
from ott2.neural.methods.flows.dynamics import LagrangianFlow
from IPython.display import clear_output

from flax.struct import PyTreeNode

In [39]:
class LagrangianPotentialFree(PyTreeNode):
    @abstractmethod
    def __call__(self, x):
        return 0.

## Gaussians (high dimensional)

In [ ]:
%load_ext autoreload
%autoreload 2

# dataset params
ds_size = 20_000
# ds_dim = 3 * 64**2
ds_dim = 2
sigma = 0.1

# training params
batch_size = 512
n_iters = 1_000_000
collect_buffer_iters = 10_000
update_potential_every = 4


src_mu = -np.ones(ds_dim)
trg_mu = np.ones(ds_dim)

ot_loader = OTLoader(
    src_ds=np.random.randn(ds_size, ds_dim) * sigma + src_mu,
    trg_ds=np.random.randn(ds_size, ds_dim) * sigma + trg_mu,
    shuffle=True,
    batch_size=batch_size,
    drop_last=True,
)
ot_loader = iter(ot_loader)

potential_data_loader = iter(ot_loader)
potential = LagrangianPotentialFree()

def callback(step, training_logs, transport):
    clear_output()
    pi0 = next(potential_data_loader)['src_lin']
    pi1 = next(potential_data_loader)['tgt_lin']

    cost, trajs = transport(pi0)

    fig, ax = plt.subplots()
    ax.scatter(pi0[:,0], pi0[:, 1], c='red', alpha=1, s=4)
    ax.scatter(pi1[:,0], pi1[:, 1], c='black', alpha=0.5, s=10)
    ax.scatter(trajs[-1].x[:, 0], trajs[-1].x[:, 1], c='green', s=10)
    ax.set_xlim((-1.5, 1.5))
    ax.set_ylim((-1.5, 1.5))
    draw_trajs(trajs=jnp.stack([traj.x for traj in trajs])[:, :100], ax=ax)
    plt.show()
     
net = MLP(hidden_layers=[256, 256, 256, 1])
num_iterations = n_iters
noc = NeuralOC(
    input_dim=ds_dim, 
    value_model=net, 
    optimizer=optax.adam(learning_rate=1e-4), 
    control_steps=30,
    reg_weight=0.,
    control_weight=1., 
    acc_weight=0.,
    potential_weight=0., 
    flow=LagrangianFlow(sigma=0.1, potential=potential), 
    key=GLOBAL_KEY,
    batch_size=batch_size,
)

logs = noc(
    potential_data_loader,
    n_iters=num_iterations,
    rng=GLOBAL_KEY,
    callback=callback,
    collect_buffer_iters=collect_buffer_iters,
    update_potential_every=update_potential_every,
)

## Three Gaussians

In [ ]:
%load_ext autoreload
%autoreload 2

# dataset params
ds_size = 20_000
# ds_dim = 3 * 64**2
ds_dim = 128
sigma = 0.1

# training params
batch_size = 512
n_iters = 1_000_000
collect_buffer_iters = 1_000_000
update_potential_every = 4

src_mu = np.zeros(ds_dim)
trg_mu1 = np.ones(ds_dim)
trg_mu2 = -np.ones(ds_dim)
trg_mu3 = np.asarray([(-1)**i for i in range(ds_dim)])
trg_mu4 = np.asarray([(-1)**(i+1) for i in range(ds_dim)])

ot_loader = OTLoader(
    src_ds=np.random.randn(ds_size, ds_dim) * sigma + src_mu,
    trg_ds=np.concatenate([
        np.random.randn(ds_size//4, ds_dim) * sigma + trg_mu1,
        np.random.randn(ds_size//4, ds_dim) * sigma + trg_mu2,
        np.random.randn(ds_size//4, ds_dim) * sigma + trg_mu3,
        np.random.randn(ds_size//4, ds_dim) * sigma + trg_mu4,
    ]),
    shuffle=True,
    batch_size=batch_size,
    drop_last=True,
)
ot_loader = iter(ot_loader)

potential_data_loader = iter(ot_loader)
potential = LagrangianPotentialFree()

def callback(step, training_logs, transport):
    clear_output()
    pi0 = next(potential_data_loader)['src_lin']
    pi1 = next(potential_data_loader)['tgt_lin']

    cost, trajs = transport(pi0)

    fig, ax = plt.subplots()
    ax.scatter(pi0[:,0], pi0[:, 1], c='red', alpha=1, s=4)
    ax.scatter(pi1[:,0], pi1[:, 1], c='black', alpha=0.5, s=10)
    ax.scatter(trajs[-1].x[:, 0], trajs[-1].x[:, 1], c='green', s=10)
    ax.set_xlim((-1.5, 1.5))
    ax.set_ylim((-1.5, 1.5))
    draw_trajs(trajs=jnp.stack([traj.x for traj in trajs])[:, :100], ax=ax)
    plt.show()

net = MLP(hidden_layers=[512, 512, 512, 512, 512, 1])
num_iterations = n_iters
noc = NeuralOC(
    input_dim=ds_dim, 
    value_model=net, 
    # optimizer=optax.adam(learning_rate=1e-4, b1=0.5, b2=0.5), 
    optimizer=optax.chain(
        optax.clip(max_delta=1.),
        optax.adam(learning_rate=1e-4, b1=0.5, b2=0.5),
    ),
    control_steps=30,
    reg_weight=0.,
    control_weight=1.,
    acc_weight=0.,
    potential_weight=0., 
    flow=LagrangianFlow(sigma=0., potential=potential), 
    key=GLOBAL_KEY,
    batch_size=batch_size,
)

logs = noc(
    potential_data_loader,
    n_iters=num_iterations,
    rng=GLOBAL_KEY,
    callback=callback,
    collect_buffer_iters=collect_buffer_iters,
    update_potential_every=update_potential_every,
)

## Images example

In [ ]:
from torch.utils.data import Subset, DataLoader, Dataset, ConcatDataset
from torchvision.transforms import Compose, Resize, Normalize, ToTensor, RandomCrop, RandomHorizontalFlip, RandomVerticalFlip, Lambda, Pad, CenterCrop, RandomResizedCrop
from torchvision.datasets import ImageFolder

class MyImageFolder(ImageFolder):
    def __getitem__(self, idx: int):
        return super().__getitem__(idx)[0].numpy()

img_size = 64

# anime dataset
path = "/home/jovyan/nazar/aligned_anime_faces"
transform = Compose([Resize((img_size, img_size)), ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
anime_dataset = MyImageFolder(path, transform=transform)

# celeba female dataset
path = "/home/jovyan/nazar/celeba_female"
attrs_path = "/home/jovyan/nazar/list_attr_celeba.txt" 
transform = Compose([Resize((img_size, img_size)), ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
celeba_female_dataset = MyImageFolder(path, transform=transform)

with open(attrs_path, 'r') as f:
    lines = f.readlines()[1:]
idx = [i for i in list(range(len(lines))) if lines[i].replace('  ', ' ').split(' ')[21] == '-1']
print("celeba", len(idx), len(lines), len(celeba_female_dataset))

celeba_female_dataset = Subset(celeba_female_dataset, idx)

In [ ]:
# training params
batch_size = 64
n_iters = 1_000_000
collect_buffer_iters = 10_000
update_potential_every = 2
eval_every = 2_000

nc, dim = 3, 64
ds_dim = nc * dim**2

#
ot_loader = OTLoader(
    src_ds=anime_dataset,
    trg_ds=celeba_female_dataset,
    flatten_flag=True,
    shuffle=True,
    batch_size=batch_size,
    num_workers=4,
    drop_last=True,
)
ot_loader = iter(ot_loader)

potential_data_loader = iter(ot_loader)
potential = LagrangianPotentialFree()

def callback(step, training_logs, transport):
    clear_output()
    pi0 = next(potential_data_loader)['src_lin']
    pi1 = next(potential_data_loader)['tgt_lin']

    cost, trajs = transport(pi0)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    def to_range(image):
        image -= image.min()
        return image / image.max()

    axes[0].imshow(to_range(pi0[0]).reshape(nc, img_size, img_size).transpose(1, 2, 0))
    axes[0].set_title('Source')
    axes[0].axis('off')

    axes[1].imshow(to_range(trajs[-1].x[0]).reshape(nc, img_size, img_size).transpose(1, 2, 0))
    axes[1].set_title('SourceMapped')
    axes[1].axis('off')

    axes[2].imshow(to_range(pi1[0]).reshape(nc, img_size, img_size).transpose(1, 2, 0))
    axes[2].set_title('Target')
    axes[2].axis('off')
    
    plt.show()
     
net = ResNetDwTime(size=dim, nc=nc)
num_iterations = n_iters
lr_schedule = optax.cosine_decay_schedule(
    init_value=2e-5, decay_steps=num_iterations, alpha=1e-2
)
noc = NeuralOC(
    input_dim=ds_dim, 
    value_model=net, 
    optimizer=optax.adam(learning_rate=lr_schedule), 
    control_steps=30,
    reg_weight=0.001,
    control_weight=0.1, 
    acc_weight=0.5,
    potential_weight=0., 
    flow=LagrangianFlow(sigma=0.1, potential=potential), 
    key=GLOBAL_KEY,
    batch_size=batch_size,
)

logs = noc(
    potential_data_loader,
    n_iters=num_iterations,
    rng=GLOBAL_KEY,
    callback=callback,
    collect_buffer_iters=collect_buffer_iters,
    update_potential_every=update_potential_every,
    eval_every=eval_every,
)